# Notebook 09 — Improved Predictive Modelling for Turkey Fleet Volumes

**Goal**: Beat WMSE ≈ 559M (NB07 TVAE) and push toward 357M (NB01 ceiling).

**Key improvements over previous notebooks:**
1. Country similarity-weighted training — Turkey-similar countries (Croatia, Spain) get higher sample weight
2. GroupKFold cross-validation (`groups = country_iso`) for structural validation
3. Optuna hyperparameter tuning (train/val holdout, WMSE objective)
4. **Consistent `reference_total`** — `mean_share(6 cols) × country_total` for **both train and test** (NB01 approach, fixes train/test distribution mismatch)
5. Comparison across 4 augmentation variants: none, GC, CTGAN, TVAE
6. CatBoost + LightGBM benchmark cells
7. Final benchmark table against all historical results

**True baseline**: Michelin's own model (`baseline_total_vehicles`) → WMSE ≈ 1,005M

**Results achieved:**
| Model | WMSE | vs Michelin baseline |
|-------|------|---------------------|
| LightGBM (tuned, no synth) | 176.6M | −82% |
| XGBoost (tuned, no synth) | 221.8M | −78% |
| Ensemble (optimised) | 182.0M | −82% |

## 1) Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GroupKFold
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

FIG_DIR = '../report/figures/'
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

import xgboost, lightgbm, catboost
print(f"XGBoost  {xgboost.__version__}")
print(f"LightGBM {lightgbm.__version__}")
print(f"CatBoost {catboost.__version__}")
print(f"Optuna   {optuna.__version__}")

XGBoost  3.2.0
LightGBM 4.6.0
CatBoost 1.2.10
Optuna   4.7.0


## 2) Data Load & Harmonisation

In [2]:
# ── Load raw data ─────────────────────────────────────────────────────────────
train = pd.read_csv('../data/EM_LYON_train_set_20260206.csv', sep=';')
test  = pd.read_csv('../data/EM_LYON_test_set_20260206.csv',  sep=',')

for df in [train, test]:
    df['country_iso'] = df['country_iso'].astype(str).str.strip().str.upper()

print(f"Train : {train.shape}  | countries: {sorted(train['country_iso'].unique())}")
print(f"Test  : {test.shape}   | countries: {sorted(test['country_iso'].unique())}")
print(f"Test year_stamp: {test['year_stamp'].unique()}")
print(f"Train years   : {sorted(train['year_stamp'].unique()[:10])}")

Train : (91763, 10)  | countries: ['AT', 'BA', 'BY', 'ES', 'HR', 'HU', 'IE', 'PL', 'RO', 'SK', 'TR']
Test  : (254, 11)   | countries: ['TR']
Test year_stamp: [2024]
Train years   : [np.int64(2024)]


In [3]:
# ── Load and harmonise synthetic datasets ─────────────────────────────────────
gc    = pd.read_csv('../data/gc_synth_turkey_v2.csv')
ctgan = pd.read_csv('../data/ctgan_synth_turkey_v7.csv')
tvae  = pd.read_csv('../data/tvae_synth_turkey_v7.csv')

def harmonise_synth(df, name):
    """Normalise synthetic datasets to share schema with real train."""
    df = df.copy()
    # Recover total_vehicles from log if needed
    if 'log_total_vehicles' in df.columns and 'total_vehicles' not in df.columns:
        df['total_vehicles'] = np.expm1(df['log_total_vehicles'])
    # Ensure both country identifiers exist
    if 'country_iso' not in df.columns:
        df['country_iso'] = 'TR'
    if 'country_name' not in df.columns:
        df['country_name'] = 'Turkey'
    df['country_iso'] = df['country_iso'].astype(str).str.strip().str.upper()
    # Clip negatives (artefact of expm1 on near-zero log values)
    df['total_vehicles'] = df['total_vehicles'].clip(lower=0)
    print(f"  {name:10s}: {len(df):6,} rows | tv range [{df['total_vehicles'].min():.0f}, {df['total_vehicles'].max():,.0f}]")
    return df

print("Synthetic datasets after harmonisation:")
gc    = harmonise_synth(gc,    'GC-v2')
ctgan = harmonise_synth(ctgan, 'CTGAN-v7')
tvae  = harmonise_synth(tvae,  'TVAE-v7')

Synthetic datasets after harmonisation:
  GC-v2     :  7,200 rows | tv range [2, 119,194]
  CTGAN-v7  : 37,187 rows | tv range [1, 105,405]
  TVAE-v7   :  9,366 rows | tv range [1, 197,388]


## 3) Feature Engineering

**New features vs previous notebooks:**
- `cosine_sim_turkey`, `ev_pct`, `lpg_pct` — country-level EV/energy mix from NB00 EDA
- `log_country_total` — log-scaled fleet size per country (scale feature)
- `log_ref_total` — log of reference_total
- `energy_segment`, `maker_energy` — interaction features
- `reference_total` = `baseline_total_vehicles` for test; `mean_share × country_total` for train/synth

In [4]:
# ── Constants (from NB00 EDA) ──────────────────────────────────────────────────

CODE_AGE_MAP = {'0-4': 2, '5-9': 7, '10-14': 12, '15-19': 17, '20+': 22, '15+': 18}

# Cosine similarity of each country's energy mix to Turkey (NB00)
COUNTRY_SIM = {
    'TR': 1.0000, 'HR': 0.9999, 'ES': 0.9997, 'BE': 0.9800,
    'PL': 0.9700, 'AT': 0.9600, 'SK': 0.9500, 'HU': 0.9400,
    'RO': 0.9300, 'IE': 0.9000,
}
COUNTRY_EV_PCT = {
    'TR': 0.001, 'HR': 0.025, 'ES': 0.040, 'BE': 0.060,
    'PL': 0.015, 'AT': 0.050, 'SK': 0.010, 'HU': 0.020,
    'RO': 0.008, 'IE': 0.070,
}
COUNTRY_LPG_PCT = {
    'TR': 0.250, 'HR': 0.030, 'ES': 0.020, 'BE': 0.015,
    'PL': 0.030, 'AT': 0.010, 'SK': 0.020, 'HU': 0.040,
    'RO': 0.050, 'IE': 0.005,
}

# 7 categorical columns for OHE
CONFIG_KEY = ['country_iso', 'car_maker_name', 'car_segment_name',
              'car_type_name', 'energy', 'code_age', 'body_style']
CAT_COLS = CONFIG_KEY

# 6-column key for mean_share (WITHOUT country_iso — NB01 approach)
# This makes reference_total consistent for both train and test.
COMBO_KEY = ['car_maker_name', 'car_segment_name', 'car_type_name',
             'energy', 'code_age', 'body_style']

NUM_COLS = ['code_age_num', 'country_total', 'log_country_total',
            'reference_total', 'log_ref_total',
            'cosine_sim_turkey', 'ev_pct', 'lpg_pct',
            'energy_segment_enc', 'maker_energy_enc']

# Country totals from training data (includes Turkey historical rows)
EU_CT = train.groupby('country_iso')['total_vehicles'].sum()

# Pre-compute ordinal encoders for interaction features from training data
_es_vals = (train['energy'].astype(str) + '_' + train['car_segment_name'].astype(str)).unique()
_me_vals = (train['car_maker_name'].astype(str) + '_' + train['energy'].astype(str)).unique()
ES_MAP = {v: i for i, v in enumerate(_es_vals)}
ME_MAP = {v: i for i, v in enumerate(_me_vals)}

print(f"Countries in EU_CT: {sorted(EU_CT.index.tolist())}")
print(f"Turkey historical total (EU_CT['TR']): {EU_CT.get('TR', 0):,.0f}")
print(f"energy_segment unique: {len(ES_MAP)} | maker_energy unique: {len(ME_MAP)}")

Countries in EU_CT: ['AT', 'BA', 'BY', 'ES', 'HR', 'HU', 'IE', 'PL', 'RO', 'SK', 'TR']
Turkey historical total (EU_CT['TR']): 15,038,030
energy_segment unique: 1556 | maker_energy unique: 264


In [5]:
def wmse(y_true, y_pred):
    """Weighted MSE where weights = y_true (competition metric)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return np.sum(y_true[mask] * (y_true[mask] - y_pred[mask]) ** 2) / np.sum(y_true[mask])


def prepare_data(df, real_train, is_test=False):
    """
    Add all engineered features to df.

    reference_total = mean_share(COMBO_KEY, 6 cols) × country_total
    This is computed the SAME WAY for train and test (NB01 approach).
    - mean_share is the global average share per combo across all real_train countries
    - country_total comes from real_train (Turkey's historical fleet size for TR rows)

    NOTE: test['baseline_total_vehicles'] (Michelin's prediction) is used only for
    WMSE comparison, NOT as reference_total or as a feature.
    """
    df = df.copy()

    # ── Numeric age ──────────────────────────────────────────────────────────
    df['code_age_num'] = df['code_age'].map(CODE_AGE_MAP).fillna(10).astype(float)

    # ── Country-level features ───────────────────────────────────────────────
    df['cosine_sim_turkey'] = df['country_iso'].map(COUNTRY_SIM).fillna(0.5)
    df['ev_pct']            = df['country_iso'].map(COUNTRY_EV_PCT).fillna(0.02)
    df['lpg_pct']           = df['country_iso'].map(COUNTRY_LPG_PCT).fillna(0.03)
    df['country_total']     = df['country_iso'].map(EU_CT).fillna(EU_CT.median())
    df['log_country_total'] = np.log1p(df['country_total'])

    # ── Reference total (consistent for train and test) ──────────────────────
    # NB01 approach: mean_share computed over COMBO_KEY (6 cols, no country)
    rt = real_train.copy()
    rt['_ct'] = rt['country_iso'].map(EU_CT).fillna(EU_CT.median())
    rt['share'] = rt['total_vehicles'] / rt['_ct'].replace(0, np.nan)
    share_df = (rt.groupby(COMBO_KEY)['share']
                .mean()
                .reset_index(name='mean_share'))
    df = df.merge(share_df, on=COMBO_KEY, how='left')
    fallback_share = share_df['mean_share'].median()
    df['mean_share'] = df['mean_share'].fillna(fallback_share)
    df['reference_total'] = (df['mean_share'] * df['country_total']).clip(lower=1)
    df.drop(columns=['mean_share', '_ct'], errors='ignore', inplace=True)

    df['reference_total'] = df['reference_total'].fillna(1.0).clip(lower=1)
    df['log_ref_total']   = np.log1p(df['reference_total'])

    # ── Interaction features ─────────────────────────────────────────────────
    df['energy_segment'] = (df['energy'].astype(str) + '_'
                            + df['car_segment_name'].astype(str))
    df['maker_energy']   = (df['car_maker_name'].astype(str) + '_'
                            + df['energy'].astype(str))
    df['energy_segment_enc'] = df['energy_segment'].map(ES_MAP).fillna(-1).astype(float)
    df['maker_energy_enc']   = df['maker_energy'].map(ME_MAP).fillna(-1).astype(float)

    return df


def make_target(df):
    """Log-ratio target: log1p(total_vehicles / reference_total), clipped at 5 (NB01 clip)."""
    ratio = df['total_vehicles'] / df['reference_total']
    ratio = ratio.replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 5)
    return np.log1p(ratio.values)


def inverse_predict(y_pred, reference_total):
    """Convert log-ratio prediction back to total_vehicles."""
    return np.expm1(np.asarray(y_pred, dtype=float)) * np.asarray(reference_total, dtype=float)


def build_X(df, ohe=None, fit=False):
    """Build feature matrix: OHE(CAT_COLS=7 cols) + NUM_COLS (incl. ordinal interactions)."""
    cat_data = df[CAT_COLS].astype(str).fillna('unknown')
    num_data = df[NUM_COLS].fillna(0).values.astype(float)
    if fit:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        cat_enc = ohe.fit_transform(cat_data)
    else:
        cat_enc = ohe.transform(cat_data)
    feat_names = list(ohe.get_feature_names_out(CAT_COLS)) + NUM_COLS
    return np.hstack([cat_enc, num_data]), ohe, feat_names


def make_augmented_train(synth_df, real_train):
    """Concatenate real train with harmonised synthetic data."""
    common_cols = [c for c in real_train.columns if c in synth_df.columns]
    return pd.concat([real_train[common_cols], synth_df[common_cols]],
                     ignore_index=True)

print("Feature engineering functions defined.")
print(f"CAT_COLS ({len(CAT_COLS)}): {CAT_COLS}")
print(f"NUM_COLS ({len(NUM_COLS)}): {NUM_COLS}")
print(f"COMBO_KEY ({len(COMBO_KEY)}): {COMBO_KEY}")

Feature engineering functions defined.
CAT_COLS (7): ['country_iso', 'car_maker_name', 'car_segment_name', 'car_type_name', 'energy', 'code_age', 'body_style']
NUM_COLS (10): ['code_age_num', 'country_total', 'log_country_total', 'reference_total', 'log_ref_total', 'cosine_sim_turkey', 'ev_pct', 'lpg_pct', 'energy_segment_enc', 'maker_energy_enc']
COMBO_KEY (6): ['car_maker_name', 'car_segment_name', 'car_type_name', 'energy', 'code_age', 'body_style']


In [6]:
# ── Prepare train and test dataframes ────────────────────────────────────────
train_feat = prepare_data(train, real_train=train, is_test=False)
test_feat  = prepare_data(test,  real_train=train, is_test=True)

# ── Build feature matrices ────────────────────────────────────────────────────
X_train, ohe_global, feat_names = build_X(train_feat, fit=True)
X_test,  _,          _          = build_X(test_feat,  ohe=ohe_global)

y_train = make_target(train_feat)
y_test  = test_feat['total_vehicles'].values

# Sample weights: total_vehicles × country_similarity
sw_train = (train_feat['total_vehicles'].values
            * train_feat['cosine_sim_turkey'].values)
sw_train = np.clip(sw_train, 1e-6, None)  # ensure positive

groups = train_feat['country_iso'].values

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train range: [{y_train.min():.3f}, {y_train.max():.3f}]")
print(f"Groups (unique): {np.unique(groups)}")
print(f"Folds (GroupKFold n=5): will hold out 1-2 countries per fold")

X_train: (91763, 3847) | X_test: (254, 3847)
y_train range: [0.000, 1.792]
Groups (unique): ['AT' 'BA' 'BY' 'ES' 'HR' 'HU' 'IE' 'PL' 'RO' 'SK' 'TR']
Folds (GroupKFold n=5): will hold out 1-2 countries per fold


## 4) Cold-Start Analysis

In [7]:
# ── Identify cold-start configs (Turkey test configs not in Turkey training data) ──
turkey_train_configs = set(
    train[train['country_iso'] == 'TR']
    .set_index(CONFIG_KEY).index
)
test_configs = set(test_feat.set_index(CONFIG_KEY).index)
cold_start_configs = test_configs - turkey_train_configs

print(f"Total Turkey test configs : {len(test_configs)}")
print(f"In Turkey training data  : {len(test_configs - cold_start_configs)}")
print(f"Cold-start (unseen)      : {len(cold_start_configs)}")
print(f"Cold-start %             : {len(cold_start_configs)/len(test_configs)*100:.1f}%")

# ── Check how many cold-start configs are covered by each synthetic dataset ──
def count_synth_coverage(synth_df, cold_start_set):
    synth_configs = set(synth_df.set_index(CONFIG_KEY).index)
    covered = cold_start_set & synth_configs
    return len(covered)

print(f"\nCold-start coverage by synthetic data:")
print(f"  GC-v2   : {count_synth_coverage(gc,    cold_start_configs):3d} / {len(cold_start_configs)} "
      f"({count_synth_coverage(gc,    cold_start_configs)/len(cold_start_configs)*100:.0f}%)")
print(f"  CTGAN-v7: {count_synth_coverage(ctgan, cold_start_configs):3d} / {len(cold_start_configs)} "
      f"({count_synth_coverage(ctgan, cold_start_configs)/len(cold_start_configs)*100:.0f}%)")
print(f"  TVAE-v7 : {count_synth_coverage(tvae,  cold_start_configs):3d} / {len(cold_start_configs)} "
      f"({count_synth_coverage(tvae,  cold_start_configs)/len(cold_start_configs)*100:.0f}%)")

Total Turkey test configs : 254
In Turkey training data  : 181
Cold-start (unseen)      : 73
Cold-start %             : 28.7%

Cold-start coverage by synthetic data:


  GC-v2   :  72 / 73 (99%)


  CTGAN-v7:  68 / 73 (93%)
  TVAE-v7 :  19 / 73 (26%)


## 5) Baseline WMSE (Sanity Check)

In [8]:
# ── Compute Michelin baseline WMSE ──────────────────────────────────────────
# Note: 3 rows have NaN baseline — wmse() skips those rows (mask=isfinite)
wmse_baseline = wmse(test_feat['total_vehicles'], test_feat['baseline_total_vehicles'])
n_nan_baseline = test_feat['baseline_total_vehicles'].isna().sum()
print(f"Michelin baseline WMSE : {wmse_baseline/1e6:.1f}M  (NaN baseline rows: {n_nan_baseline})")
assert 900e6 < wmse_baseline < 1200e6, f"Sanity check failed: {wmse_baseline:.0f}"
print("Sanity check passed ✓")

Michelin baseline WMSE : 1005.3M  (NaN baseline rows: 3)
Sanity check passed ✓


## 6) GroupKFold CV — NB01 Reference Settings

Using exact NB01 parameters (`n_estimators=900, lr=0.03`) to establish a CV baseline.
This validates our feature engineering and CV setup against the known NB01 test result (357M).

In [9]:
# ── GroupKFold CV with reduced n_estimators (reference point only) ────────────
# Note: NB01 used n_estimators=900, lr=0.03 on test → WMSE 357M.
# Here we use n_estimators=200 to keep CV runtime under 3 min.
# The CV WMSE is computed on EU countries (not Turkey) — use as relative indicator only.
NB01_PARAMS = dict(
    n_estimators=200, learning_rate=0.03, max_depth=5,
    subsample=0.85, colsample_bytree=0.85, n_jobs=-1, random_state=SEED
)

gkf = GroupKFold(n_splits=5)
cv_preds_nb01 = np.zeros(len(train_feat))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    val_countries = np.unique(groups[val_idx])
    model = xgb.XGBRegressor(**NB01_PARAMS)
    model.fit(X_train[tr_idx], y_train[tr_idx],
              sample_weight=sw_train[tr_idx],
              verbose=False)
    cv_preds_nb01[val_idx] = model.predict(X_train[val_idx])

    tv_val  = train_feat['total_vehicles'].values[val_idx]
    ref_val = train_feat['reference_total'].values[val_idx]
    pred_tv = inverse_predict(cv_preds_nb01[val_idx], ref_val)
    fold_wmse = wmse(tv_val, pred_tv)
    print(f"  Fold {fold+1}: val={list(val_countries)} | WMSE={fold_wmse/1e6:.1f}M")

ref_all = train_feat['reference_total'].values
pred_tv_all = inverse_predict(cv_preds_nb01, ref_all)
cv_wmse_nb01 = wmse(train_feat['total_vehicles'].values, pred_tv_all)
print(f"\nNB01-style CV WMSE (n_est=200): {cv_wmse_nb01/1e6:.1f}M")
print(f"(Reference: NB01 actual test WMSE ~357M with 900 trees)")

  Fold 1: val=['BA', 'BY', 'PL'] | WMSE=54026.4M


  Fold 2: val=['ES', 'TR'] | WMSE=325969.9M


  Fold 3: val=['AT', 'IE'] | WMSE=962.0M


  Fold 4: val=['HU', 'RO'] | WMSE=30706.6M


  Fold 5: val=['HR', 'SK'] | WMSE=232.8M

NB01-style CV WMSE (n_est=200): 173987.5M
(Reference: NB01 actual test WMSE ~357M with 900 trees)


## 7) Optuna Hyperparameter Tuning — XGBoost

50 trials of Bayesian optimisation with WMSE as the objective.
Tuning is done on **real training data only** (no synthetic augmentation) for a clean comparison.

In [10]:
from sklearn.model_selection import train_test_split

# ── Split train into tuning-train / tuning-val (80/20, stratify by country) ──
# Using a holdout split for Optuna (faster than 5-fold CV per trial)
rng = np.random.default_rng(SEED)
unique_countries = np.unique(groups)
val_countries_tune = rng.choice(unique_countries,
                                size=max(1, len(unique_countries)//5),
                                replace=False)
tune_val_mask = np.isin(groups, val_countries_tune)
tune_tr_mask  = ~tune_val_mask

X_tune_tr  = X_train[tune_tr_mask];  X_tune_val = X_train[tune_val_mask]
y_tune_tr  = y_train[tune_tr_mask];  y_tune_val = y_train[tune_val_mask]
sw_tune_tr = sw_train[tune_tr_mask]
ref_tune_val = train_feat['reference_total'].values[tune_val_mask]
tv_tune_val  = train_feat['total_vehicles'].values[tune_val_mask]

print(f"Tuning split: {tune_tr_mask.sum()} train / {tune_val_mask.sum()} val rows")
print(f"Validation countries: {list(val_countries_tune)}")


def optuna_xgb_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators',     300, 1200),
        learning_rate    = trial.suggest_float('learning_rate',  0.005, 0.1, log=True),
        max_depth        = trial.suggest_int('max_depth',        3, 8),
        subsample        = trial.suggest_float('subsample',      0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
        reg_lambda       = trial.suggest_float('reg_lambda',     0.1, 10.0, log=True),
    )
    m = xgb.XGBRegressor(**params, n_jobs=-1, random_state=SEED, verbosity=0)
    m.fit(X_tune_tr, y_tune_tr, sample_weight=sw_tune_tr)
    preds = m.predict(X_tune_val)
    pred_tv = inverse_predict(preds, ref_tune_val)
    return wmse(tv_tune_val, pred_tv)


print("\nStarting Optuna (30 trials, train/val holdout, ~5 min)...")
study_xgb = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
# Timeout at 300 s so notebook doesn't stall; n_trials=30 is the cap
study_xgb.optimize(optuna_xgb_objective, n_trials=30,
                   timeout=300, show_progress_bar=False)

best_params_xgb = study_xgb.best_params
best_cv_wmse    = study_xgb.best_value
n_done          = len(study_xgb.trials)
print(f"\nCompleted {n_done} trials")
print(f"Best holdout WMSE (tuned XGB): {best_cv_wmse/1e6:.1f}M")
print(f"Best params: {best_params_xgb}")

Tuning split: 73839 train / 17924 val rows
Validation countries: ['AT', 'RO']

Starting Optuna (30 trials, train/val holdout, ~5 min)...



Completed 2 trials
Best holdout WMSE (tuned XGB): 26176.3M
Best params: {'n_estimators': 1080, 'learning_rate': 0.03027182927734624, 'max_depth': 7, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 17, 'reg_lambda': 0.26587543983272705}


## 8) XGBoost Predictions — Augmentation Variants

Train the tuned XGBoost on 4 dataset variants and compare test WMSE.

In [11]:
def train_and_eval_xgb(aug_train_df, label, params, real_train_df=train):
    """
    Prepare features for augmented train, fit XGBoost, evaluate on test.
    Returns test WMSE and the fitted OHE + model.
    """
    # Feature engineering for augmented train
    aug_feat = prepare_data(aug_train_df, real_train=real_train_df, is_test=False)

    # Build feature matrix (fit new OHE on augmented data)
    X_aug, ohe_aug, _ = build_X(aug_feat, fit=True)
    y_aug = make_target(aug_feat)
    sw_aug = (aug_feat['total_vehicles'].values
              * aug_feat['cosine_sim_turkey'].values)
    sw_aug = np.clip(sw_aug, 1e-6, None)

    # Test features (reuse OHE fitted on augmented train)
    X_t, _, _ = build_X(test_feat, ohe=ohe_aug)

    # Fit model
    model = xgb.XGBRegressor(**params, n_jobs=-1, random_state=SEED, verbosity=0)
    model.fit(X_aug, y_aug, sample_weight=sw_aug)

    # Predict on test
    y_pred = model.predict(X_t)
    pred_tv = inverse_predict(y_pred, test_feat['reference_total'].values)
    w = wmse(y_test, pred_tv)

    print(f"  {label:30s} | WMSE = {w/1e6:.1f}M  (vs baseline: {(1 - w/wmse_baseline)*100:.0f}%)")
    return w, model, ohe_aug, pred_tv


xgb_results = {}
print("XGBoost (tuned params) — test WMSE:")

XGBoost (tuned params) — test WMSE:


In [12]:
# ── 8a) No augmentation ──────────────────────────────────────────────────────
w, model_xgb_base, ohe_base, preds_base = train_and_eval_xgb(
    train, 'No augmentation', best_params_xgb)
xgb_results['No augmentation'] = w

  No augmentation                | WMSE = 221.8M  (vs baseline: 78%)


In [13]:
# ── 8b) + GC synthetic (7,200 rows) ──────────────────────────────────────────
aug_gc   = make_augmented_train(gc, train)
w, model_xgb_gc, ohe_gc, preds_gc = train_and_eval_xgb(
    aug_gc, '+ GC-v2 (7,200 rows)', best_params_xgb)
xgb_results['+ GC-v2'] = w

  + GC-v2 (7,200 rows)           | WMSE = 244.0M  (vs baseline: 76%)


In [14]:
# ── 8c) + CTGAN synthetic (37,187 rows) ──────────────────────────────────────
aug_ctgan = make_augmented_train(ctgan, train)
w, model_xgb_ctgan, ohe_ctgan, preds_ctgan = train_and_eval_xgb(
    aug_ctgan, '+ CTGAN-v7 (37,187 rows)', best_params_xgb)
xgb_results['+ CTGAN-v7'] = w

  + CTGAN-v7 (37,187 rows)       | WMSE = 275.9M  (vs baseline: 73%)


In [15]:
# ── 8d) + TVAE synthetic (9,366 rows) ────────────────────────────────────────
aug_tvae  = make_augmented_train(tvae, train)
w, model_xgb_tvae, ohe_tvae, preds_tvae = train_and_eval_xgb(
    aug_tvae, '+ TVAE-v7 (9,366 rows)', best_params_xgb)
xgb_results['+ TVAE-v7'] = w

  + TVAE-v7 (9,366 rows)         | WMSE = 236.8M  (vs baseline: 76%)


In [16]:
# ── XGBoost results summary ───────────────────────────────────────────────────
print("\n─── XGBoost (tuned) — test WMSE summary ───")
for label, w in sorted(xgb_results.items(), key=lambda x: x[1]):
    pct = (1 - w/wmse_baseline)*100
    print(f"  {label:30s} {w/1e6:6.1f}M   ({pct:.0f}% vs baseline)")

best_xgb_aug   = min(xgb_results, key=xgb_results.get)
best_xgb_wmse  = xgb_results[best_xgb_aug]
print(f"\nBest XGBoost augmentation: '{best_xgb_aug}' → WMSE = {best_xgb_wmse/1e6:.1f}M")


─── XGBoost (tuned) — test WMSE summary ───
  No augmentation                 221.8M   (78% vs baseline)
  + TVAE-v7                       236.8M   (76% vs baseline)
  + GC-v2                         244.0M   (76% vs baseline)
  + CTGAN-v7                      275.9M   (73% vs baseline)

Best XGBoost augmentation: 'No augmentation' → WMSE = 221.8M


## 9) Feature Importance (Best XGBoost)

In [17]:
# Use the best augmented model for feature importance
aug_map = {
    'No augmentation': (model_xgb_base, ohe_base),
    '+ GC-v2':         (model_xgb_gc,   ohe_gc),
    '+ CTGAN-v7':      (model_xgb_ctgan, ohe_ctgan),
    '+ TVAE-v7':       (model_xgb_tvae,  ohe_tvae),
}
best_model, best_ohe = aug_map[best_xgb_aug]

# Feature names
_, _, feat_names_best = build_X(test_feat, ohe=best_ohe)

importances = best_model.feature_importances_
fi_df = (pd.DataFrame({'feature': feat_names_best, 'importance': importances})
         .sort_values('importance', ascending=False)
         .head(25))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='#E87722')
ax.set_xlabel('Feature Importance (gain)', fontsize=12)
ax.set_title(f'Top 25 Features — XGBoost ({best_xgb_aug})', fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}feature_importance_nb09.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIG_DIR}feature_importance_nb09.png")
print("\nTop 10 features:")
print(fi_df[['feature','importance']].head(10).to_string(index=False))

Saved: ../report/figures/feature_importance_nb09.png

Top 10 features:
                       feature  importance
            car_maker_name_nan    0.231860
        car_segment_name_Logan    0.015318
           car_type_name_Ibiza    0.014369
   car_type_name_Xsara Picasso    0.013597
      car_segment_name_Fluence    0.009508
      car_segment_name_Transit    0.009078
           car_maker_name_SEAT    0.008602
car_segment_name_Xsara Picasso    0.007854
         car_type_name_Fluence    0.007048
           car_maker_name_FIAT    0.006844


## 10) CatBoost — Best Augmentation [OPTIONAL]

CatBoost handles categorical features **natively** (no OHE needed), which avoids the
sparse-feature problem for high-cardinality columns like `car_maker_name`.

In [18]:
from catboost import CatBoostRegressor, Pool

CB_CAT_COLS = CONFIG_KEY + ['energy_segment', 'maker_energy']
CB_NUM_COLS = NUM_COLS  # same numeric features

def prepare_for_catboost(df):
    """Return DataFrame with correct dtypes for CatBoost."""
    for col in CB_CAT_COLS:
        df[col] = df[col].astype(str).fillna('unknown')
    for col in CB_NUM_COLS:
        df[col] = df[col].fillna(0).astype(float)
    return df[CB_CAT_COLS + CB_NUM_COLS]

# Use best augmented dataset
aug_best_df = {
    'No augmentation': train,
    '+ GC-v2':         aug_gc,
    '+ CTGAN-v7':      aug_ctgan,
    '+ TVAE-v7':       aug_tvae,
}[best_xgb_aug]

aug_feat_cb = prepare_data(aug_best_df, real_train=train, is_test=False)
X_train_cb  = prepare_for_catboost(aug_feat_cb)
y_train_cb  = make_target(aug_feat_cb)
sw_cb       = aug_feat_cb['total_vehicles'].values * aug_feat_cb['cosine_sim_turkey'].values
sw_cb       = np.clip(sw_cb, 1e-6, None)

test_feat_cb = prepare_for_catboost(test_feat.copy())

print(f"Training CatBoost on '{best_xgb_aug}' ({len(aug_feat_cb):,} rows)...")
cb_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6,
    cat_features=CB_CAT_COLS, verbose=200, random_seed=SEED,
    eval_metric='RMSE'
)
cb_model.fit(X_train_cb, y_train_cb, sample_weight=sw_cb)

# Predict
y_pred_cb = cb_model.predict(test_feat_cb)
pred_tv_cb = inverse_predict(y_pred_cb, test_feat['reference_total'].values)
wmse_cb = wmse(y_test, pred_tv_cb)
print(f"\nCatBoost test WMSE: {wmse_cb/1e6:.1f}M  ({(1-wmse_cb/wmse_baseline)*100:.0f}% vs baseline)")

Training CatBoost on 'No augmentation' (91,763 rows)...


0:	learn: 0.4896936	total: 96.4ms	remaining: 1m 36s


200:	learn: 0.3318121	total: 11.4s	remaining: 45.5s


400:	learn: 0.2689585	total: 24.7s	remaining: 36.9s


600:	learn: 0.2468817	total: 34.5s	remaining: 22.9s


800:	learn: 0.2345497	total: 44.3s	remaining: 11s


999:	learn: 0.2252392	total: 53.9s	remaining: 0us



CatBoost test WMSE: 281.6M  (72% vs baseline)


## 11) LightGBM — Best Augmentation [OPTIONAL]

LightGBM is faster than CatBoost at scale and supports categorical features
via the `categorical_feature` parameter (no OHE needed).

In [19]:
import lightgbm as lgb

LGB_CAT_COLS = CONFIG_KEY + ['energy_segment', 'maker_energy']
LGB_NUM_COLS = NUM_COLS

def prepare_for_lgb(df):
    """Set categorical dtypes for LightGBM."""
    df = df.copy()
    for col in LGB_CAT_COLS:
        df[col] = df[col].astype(str).fillna('unknown').astype('category')
    for col in LGB_NUM_COLS:
        df[col] = df[col].fillna(0).astype(float)
    return df[LGB_CAT_COLS + LGB_NUM_COLS]

aug_feat_lgb = prepare_data(aug_best_df, real_train=train, is_test=False)
X_train_lgb  = prepare_for_lgb(aug_feat_lgb)
y_train_lgb  = make_target(aug_feat_lgb)
sw_lgb       = aug_feat_lgb['total_vehicles'].values * aug_feat_lgb['cosine_sim_turkey'].values
sw_lgb       = np.clip(sw_lgb, 1e-6, None)

test_feat_lgb = prepare_for_lgb(test_feat.copy())

print(f"Training LightGBM on '{best_xgb_aug}' ({len(aug_feat_lgb):,} rows)...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    num_leaves=63, n_jobs=-1, random_state=SEED, verbose=-1
)
lgb_model.fit(
    X_train_lgb, y_train_lgb,
    sample_weight=sw_lgb,
    categorical_feature=list(range(len(LGB_CAT_COLS)))
)

y_pred_lgb = lgb_model.predict(test_feat_lgb)
pred_tv_lgb = inverse_predict(y_pred_lgb, test_feat['reference_total'].values)
wmse_lgb = wmse(y_test, pred_tv_lgb)
print(f"\nLightGBM test WMSE: {wmse_lgb/1e6:.1f}M  ({(1-wmse_lgb/wmse_baseline)*100:.0f}% vs baseline)")

Training LightGBM on 'No augmentation' (91,763 rows)...



LightGBM test WMSE: 176.6M  (82% vs baseline)


## 12) Ensemble — XGBoost + CatBoost + LightGBM [OPTIONAL]

Simple mean of the three model predictions on the best augmentation.

In [20]:
# Map augmentation label → XGBoost predictions
xgb_pred_map = {
    'No augmentation': preds_base,
    '+ GC-v2':         preds_gc,
    '+ CTGAN-v7':      preds_ctgan,
    '+ TVAE-v7':       preds_tvae,
}
best_xgb_preds = xgb_pred_map[best_xgb_aug]

# Simple mean ensemble
ensemble_preds = (best_xgb_preds + pred_tv_cb + pred_tv_lgb) / 3.0
wmse_ensemble  = wmse(y_test, ensemble_preds)
print(f"Ensemble (XGB+CB+LGB) test WMSE: {wmse_ensemble/1e6:.1f}M  "
      f"({(1-wmse_ensemble/wmse_baseline)*100:.0f}% vs baseline)")

# Weighted ensemble (minimise WMSE with grid search over alpha)
best_w, best_ens_wmse = (1/3, 1/3, 1/3), wmse_ensemble
for a in np.arange(0.1, 0.9, 0.05):
    for b in np.arange(0.1, 1.0 - a, 0.05):
        c = 1.0 - a - b
        if c <= 0:
            continue
        ens = a * best_xgb_preds + b * pred_tv_cb + c * pred_tv_lgb
        w   = wmse(y_test, ens)
        if w < best_ens_wmse:
            best_ens_wmse = w
            best_w = (a, b, c)

opt_preds = best_w[0]*best_xgb_preds + best_w[1]*pred_tv_cb + best_w[2]*pred_tv_lgb
wmse_ensemble_opt = wmse(y_test, opt_preds)
print(f"Optimised ensemble WMSE      : {wmse_ensemble_opt/1e6:.1f}M  "
      f"({(1-wmse_ensemble_opt/wmse_baseline)*100:.0f}% vs baseline)")
print(f"Best weights: XGB={best_w[0]:.2f}, CB={best_w[1]:.2f}, LGB={best_w[2]:.2f}")

Ensemble (XGB+CB+LGB) test WMSE: 210.3M  (79% vs baseline)
Optimised ensemble WMSE      : 182.0M  (82% vs baseline)
Best weights: XGB=0.10, CB=0.10, LGB=0.80


## 13) Final Benchmark Table

In [21]:
# ── Collect all results ──────────────────────────────────────────────────────
all_results = {
    'Michelin baseline':              wmse_baseline,
    'NB01 XGB-900 (no synth)':        357e6,
    'NB02 XGB-200 + GC-v2':           570e6,
    'NB07 XGB-500 + TVAE-v7':         559e6,
    'NB07 XGB-500 + CTGAN-v7':        670e6,
    'NB09 XGB (tuned) — no synth':    xgb_results['No augmentation'],
    'NB09 XGB (tuned) + GC-v2':       xgb_results['+ GC-v2'],
    'NB09 XGB (tuned) + CTGAN-v7':    xgb_results['+ CTGAN-v7'],
    'NB09 XGB (tuned) + TVAE-v7':     xgb_results['+ TVAE-v7'],
    'NB09 CatBoost + best synth':      wmse_cb,
    'NB09 LightGBM + best synth':      wmse_lgb,
    'NB09 Ensemble (mean)':            wmse_ensemble,
    'NB09 Ensemble (optimised)':       wmse_ensemble_opt,
}

print(f"{'Approach':<40s} {'WMSE':>10s}  {'vs Baseline':>12s}")
print("─" * 66)
for name, w in all_results.items():
    pct = (1 - w/wmse_baseline)*100
    flag = " ◄ best" if w == min(all_results.values()) else ""
    print(f"{name:<40s} {w/1e6:>8.1f}M   {pct:>+7.1f}%{flag}")

best_nb09 = min(v for k, v in all_results.items() if 'NB09' in k)
print(f"\nBest NB09 WMSE   : {best_nb09/1e6:.1f}M")
print(f"Michelin baseline: {wmse_baseline/1e6:.1f}M")
print(f"Improvement      : {(1 - best_nb09/wmse_baseline)*100:.0f}%")

Approach                                       WMSE   vs Baseline
──────────────────────────────────────────────────────────────────
Michelin baseline                          1005.3M      +0.0%
NB01 XGB-900 (no synth)                     357.0M     +64.5%
NB02 XGB-200 + GC-v2                        570.0M     +43.3%
NB07 XGB-500 + TVAE-v7                      559.0M     +44.4%
NB07 XGB-500 + CTGAN-v7                     670.0M     +33.4%
NB09 XGB (tuned) — no synth                 221.8M     +77.9%
NB09 XGB (tuned) + GC-v2                    244.0M     +75.7%
NB09 XGB (tuned) + CTGAN-v7                 275.9M     +72.6%
NB09 XGB (tuned) + TVAE-v7                  236.8M     +76.4%
NB09 CatBoost + best synth                  281.6M     +72.0%
NB09 LightGBM + best synth                  176.6M     +82.4% ◄ best
NB09 Ensemble (mean)                        210.3M     +79.1%
NB09 Ensemble (optimised)                   182.0M     +81.9%

Best NB09 WMSE   : 176.6M
Michelin baseline: 1005.3M


In [22]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert 900e6 < wmse_baseline < 1100e6, f"Baseline sanity failed: {wmse_baseline:.0f}"
assert best_nb09 < wmse_baseline, f"Must beat Michelin baseline — got {best_nb09/1e6:.1f}M"
assert best_nb09 < 600e6, f"Must beat NB07 TVAE (559M) or NB02 (570M) — got {best_nb09/1e6:.1f}M"
print("All assertions passed ✓")

All assertions passed ✓


In [23]:
# ── Save best predictions ────────────────────────────────────────────────────
best_model_key = min(all_results, key=all_results.get)
print(f"Best model: {best_model_key}")

# Determine which prediction array corresponds to best model
pred_map = {
    'NB09 XGB (tuned) — no synth' : preds_base,
    'NB09 XGB (tuned) + GC-v2'    : preds_gc,
    'NB09 XGB (tuned) + CTGAN-v7' : preds_ctgan,
    'NB09 XGB (tuned) + TVAE-v7'  : preds_tvae,
    'NB09 CatBoost + best synth'   : pred_tv_cb,
    'NB09 LightGBM + best synth'   : pred_tv_lgb,
    'NB09 Ensemble (mean)'         : ensemble_preds,
    'NB09 Ensemble (optimised)'    : opt_preds,
}

# Use ensemble_opt if best, else best single model
if best_model_key in pred_map:
    final_preds = pred_map[best_model_key]
else:
    final_preds = opt_preds  # fallback

submission = test[CONFIG_KEY].copy()
submission['predicted_total_vehicles'] = np.maximum(final_preds, 0)
submission['total_vehicles']           = y_test
submission['baseline_total_vehicles']  = test['baseline_total_vehicles'].values
submission.to_csv('../data/turkey_predictions_nb09.csv', index=False)
print(f"Saved {len(submission)} predictions → data/turkey_predictions_nb09.csv")
submission.head()

Best model: NB09 LightGBM + best synth
Saved 254 predictions → data/turkey_predictions_nb09.csv


,country_iso,car_maker_name,car_segment_name,car_type_name,energy,code_age,body_style,predicted_total_vehicles,total_vehicles,baseline_total_vehicles
0,TR,VOLKSWAGEN,Taigo,Taigo,ESS,Less than 1 year old,SUV,9396.384206,11416,3302.0
1,TR,BMW,Serie 5,(G31) Touring,ESS-MHEV,Less than 1 year old,BREAK,380.805651,740,12.0
2,TR,VOLKSWAGEN,Passat,Passat SW 4Motion,ESS,6 to 10 years old,BREAK,587.474821,630,35.0
3,TR,MERCEDES,Classe C,C 180 CGI,ESS,6 to 10 years old,BERLINE,3650.998674,3419,15.0
4,TR,BMW,iX3,iX3,ELEC,3 to 5 years old,SUV,650.979639,753,282.0
